# 30 — Leaderboard: every model and technique

Collates `ml/reports/runs/*.json` into one table and picks the model to promote to test.

**The rule this notebook enforces:** models are ranked on **dev**, and only the winner is refit on
train+dev and scored once on test. Every result file stamps the split sha, and `results.load_all()`
drops rows recorded against a different one, so a stale run cannot quietly enter the ranking.

**And the rule it enforces second:** rank against the confidence interval, not the point estimate.
Dev holds 68 unique Negative tickets, so the sentiment CI is roughly ±0.08. In the original
bake-off all ten of the top ten configurations sat inside one interval — their ordering was noise.
Any "winner" here whose CI overlaps the runner-up's is a tie, and should be broken on serving cost,
not on the fourth decimal place.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES
POS = config.SENTIMENT_POSITIVE_CLASS
print("split sha:", splits.sha())

In [ ]:
runs = sb.results.load_all()
print(f"{len(runs)} runs recorded against split {splits.sha()}")
if runs.empty:
    raise SystemExit("no runs found")

runs["family"] = runs.get("family", pd.Series(["classical"] * len(runs))).fillna("classical")
display(runs.groupby(["task", "family", "eval_portion"]).size().rename("runs").reset_index())

## 1. Sentiment — dev

In [ ]:
cols = ["model", "arm", "family", "eval_lang", "headline", "accuracy",
        "negative_precision", "negative_recall", "best_epoch", "train_seconds", "author"]

sent = runs[(runs.task == "sentiment") & (runs.eval_portion == "dev")]
sent = sent[[c for c in cols if c in sent.columns]].sort_values("headline", ascending=False)
display(sent.head(20))

## 2. Priority — dev

In [ ]:
prio = runs[(runs.task == "priority") & (runs.eval_portion == "dev")]
prio = prio[[c for c in cols if c in prio.columns]].sort_values("headline", ascending=False)
display(prio.head(15))

## 3. Techniques

Technique results are ablations rather than models, so they live in their own CSVs rather than in
`runs/`. Pulled together here so the whole picture sits on one page.

In [ ]:
REPORTS = REPO / "ml" / "reports"
for name, path in [
    ("lexicon correction",   REPORTS / "technique_lexicon_ablation.csv"),
    ("strategy A",           REPORTS / "technique_strategy_a.csv"),
    ("augmentation",         REPORTS / "technique_augmentation.csv"),
    ("word tokenizer",       REPORTS / "word_tokenizer_comparison.csv"),
]:
    print(f"--- {name} ---")
    if path.exists():
        display(pd.read_csv(path).head(12))
    else:
        print("   not run yet\n")

## 4. What gets promoted to test

In [ ]:
TEST_CLASSICAL = {"sentiment": 0.4572, "priority": 0.8722}   # from 10_final_test_eval.ipynb

for task in ["sentiment", "priority"]:
    d = runs[(runs.task == task) & (runs.eval_portion == "dev")]
    if d.empty:
        continue
    best = d.loc[d.headline.idxmax()]
    print(f"{task}: best on dev = {best.model} / {best.arm} ({best.family}) "
          f"{best.headline:.4f}")
    print(f"   classical dev  {d[d.family == 'classical'].headline.max():.4f}")
    print(f"   classical test {TEST_CLASSICAL[task]:.4f}   <- the number to beat in production")
    print()

print("Promote only if the encoder clears the classical dev score by more than the CI width")
print("(~0.08 for sentiment). Otherwise ship classical -- it is faster, cheaper and already built.")

## 5. Reading this table honestly

Three traps, all of which this project has already fallen into once:

1. **Dev-to-test does not hold.** `10_final_test_eval.ipynb` measured the sentiment champion
   dropping 0.6144 → 0.4572. Roughly 0.059 of that is test's lower Negative prevalence (3.28% vs
   4.54%); the remaining ~0.098 is genuine generalization loss. Expect any encoder here to lose
   ground on test too.
2. **Threshold tuning did not transfer.** CV promised +0.006 to +0.011; on test the tuned threshold
   made the SVM *worse* (0.4572 → 0.4524). Do not promote a model on a tuned-threshold number.
3. **The label ceiling.** Priority already scores 0.8722 against labels that agree with human
   judgement at 0.7722, so it is fitting the labeling rule, not human priority. Sentiment sits
   inside its own ceiling's CI. Beyond that point, better scores mean a better *labeler* imitation
   — which is an argument for the v6 relabel, not for a bigger model.